### Importing Libraries

In [1]:

import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import os
import sys
from sklearn.metrics.cluster import adjusted_rand_score
import tensorflow.compat.v1 as tf
from sklearn.preprocessing import normalize
tf.__version__

# the location of R (used for the mclust clustering)
os.environ['R_HOME'] = "/usr/lib/R"

import SynthST as SynthST
from SynthST.Train_SynthST import train_SynthST
from SynthST.utils import Cal_Spatial_Net, Stats_Spatial_Net, mclust_R, Cal_Spatial_Net_3D

2025-05-20 18:44:19.986389: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-20 18:44:20.605905: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


### Initialization

In [2]:

"""
The set of methods that participate in the simulation is determined by visualizing the CTPs 
and selecting the ones with minimal difference from the median for all the celltypes.
The list of participating methods for all the datasets in in Supplementary Table.
Training is done on output from all the methods; this list is shuffled after each epoch.
Test data: Average/Median of all the methods (no participation in training).
"""

id  = "151508" # Dataset name
data_path = "/data/Ajita/Spatial/Datasets/Spatial_Deconvolution/DLPFC/ST/" + id + "/"

results_ctp_path = "/data/Ajita/Spatial/Datasets/Spatial_Deconvolution/DLPFC/simulator_outputs/"
input_ctp_path = "/data/Ajita/Spatial/Datasets/Spatial_Deconvolution/DLPFC/deconvolution_outputs/"

methods = ["cell2location", "RCTD", "Tangram", "Stereoscope"]


In [3]:
# Reading datasets

adata = sc.read_visium(data_path, count_file = id + "_filtered_feature_bc_matrix.h5")
adata.obs_names_make_unique()
adata.var_names = pd.Categorical(adata.var_names).astype(str)
adata.var_names_make_unique()

def prepare_anndata(dt, adata):
    adata_X = sc.AnnData(dt)

    adata_X.obs_names = adata.obs_names    
    for i in adata.uns.keys():
        adata_X.uns[i] = adata.uns[i]

    for i in adata.obs.keys():
        adata_X.obs[i] = adata.obs[i]

    for i in adata.var.keys():
        adata_X.var[i] = adata.var[i]

    for i in adata.obsm.keys():
        adata_X.obsm[i] = adata.obsm[i]

    return adata_X
    

### Some Preprocessing on the input files

In [4]:
# Filter anndata based on bar_codes present in the outputs.

adata_list = []
test_data_list = {}

indices = adata.obs_names
print (len(indices))

for m in methods:
    dt = pd.read_csv(input_ctp_path + "output_" + m + id + ".csv")
    ind = dt[dt.columns[0]]
    indices = set(indices).intersection(ind)
    
filtered_adata = adata[adata.obs_names.isin(indices), :].copy()

print (filtered_adata.shape)
adata = filtered_adata


4384
(4382, 33538)


In [5]:

# Find out columns/celltype with low representation
low_prop_cols = []
    
for m in methods:
    
    dt = pd.read_csv(input_ctp_path + "output_" + m + id + ".csv")
    dt = dt[dt[dt.columns[0]].isin(indices)]    
    dt = dt.iloc[:,1:] # sorting will ensure that the columns are always sorted alphabetically.
    dt = dt.sort_index(axis = 1)
    cols = dt.columns   
    dt = dt.to_numpy() #.transpose()
    dt = normalize(dt, axis = 1, norm = "l1")

    for col in cols:
        df = pd.DataFrame(dt, columns = cols)
        if (np.sum(df[col]) <= 10):
            if col not in low_prop_cols:
                low_prop_cols += [col]
            print (m, " method", col, "==", np.sum(df[col]), "\n")
    
    test_data_list[m] = dt
    adata_X = prepare_anndata(dt, adata)
    adata_list += [adata_X] 

list_of_data = list(test_data_list.values())
arrays_stacked = np.stack(list_of_data)
average_array = pd.DataFrame(np.mean(arrays_stacked, axis = 0), index = adata_X.obs.index, columns = adata_X.var.index)
test_data_list["Average"] = average_array  
print ("Low proportion columns", low_prop_cols)


RCTD  method L5_6_CC == 8.218973669854869 

RCTD  method Neu_mat == 6.635863456461761 

Low proportion columns ['L5_6_CC', 'Neu_mat']


### Running SynthST

In [6]:
tf.compat.v1.disable_eager_execution()

for adata in adata_list:

    SynthST.Cal_Spatial_Net(adata, rad_cutoff = 150)
    #SynthST.Stats_Spatial_Net(adata)

# n_epochs = 500
adata = train_SynthST(adata_list, methods + ["Average", "Median"], alpha = 0, hidden_dims = [512, 100, 10], 
                     n_epochs = 250, save_reconstrction = True, save_attention = True)    

------Calculating spatial graph...
The graph contains 25674 edges, 4382 cells.
5.8590 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 25674 edges, 4382 cells.
5.8590 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 25674 edges, 4382 cells.
5.8590 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 25674 edges, 4382 cells.
5.8590 neighbors per cell on average.
f1 shape is: <unknown>
f1 shape is: <unknown>


2025-05-20 18:44:28.782666: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14536 MB memory:  -> device: 0, name: Quadro RTX 5000, pci bus id: 0000:af:00.0, compute capability: 7.5
2025-05-20 18:44:28.783489: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 14536 MB memory:  -> device: 1, name: Quadro RTX 5000, pci bus id: 0000:d8:00.0, compute capability: 7.5
2025-05-20 18:44:28.796297: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:375] MLIR V1 optimization pass is not enabled


Size of Input:  (4382, 17)
100%|█████████████████████████████████████████████████████████████| 500/500 [05:27<00:00,  1.53it/s]
Inference for : cell2location
[0.9999999 0.9999999 0.9999999 ... 1.0000001 1.0000001 1.0000002]
Inference for : RCTD
[0.9999998 0.9999998 0.9999998 ... 1.0000002 1.0000002 1.0000002]
Inference for : Tangram
[0.9999998 0.9999998 0.9999998 ... 1.0000001 1.0000001 1.0000001]
Inference for : Stereoscope
[0.9999998 0.9999998 0.9999998 ... 1.0000001 1.0000001 1.0000002]
Inference for : Average
[0.9999998 0.9999998 0.9999998 ... 1.0000001 1.0000002 1.0000002]
Inference for : Median
[0.9999998 0.9999998 0.9999998 ... 1.0000001 1.0000001 1.0000002]


### Writing the output

In [7]:
del adata.uns["SynthST_attention"]

for i in adata.var:
    del adata.var[i]
    
adata.var_names = cols
print (adata.obs_names)

Index(['AAACAAGTATCTCCCA-1', 'AAACAATCTACTAGCA-1', 'AAACACCAATAACTGC-1',
       'AAACAGAGCGACTCCT-1', 'AAACAGCTTTCAGAAG-1', 'AAACAGGGTCTATATT-1',
       'AAACAGTGTTCCTGGG-1', 'AAACATTTCCCGGATT-1', 'AAACCACTACACAGAT-1',
       'AAACCCGAACGAAATC-1',
       ...
       'TTGTGTTTCCCGAAAG-1', 'TTGTTAGCAAATTCGA-1', 'TTGTTCAGTGTGCTAC-1',
       'TTGTTCTAGATACGCT-1', 'TTGTTGTGTGTCAAGA-1', 'TTGTTTCACATCCAGG-1',
       'TTGTTTCATTAGTCTA-1', 'TTGTTTCCATACAACT-1', 'TTGTTTGTATTACACG-1',
       'TTGTTTGTGTAAATTC-1'],
      dtype='object', length=4382)



adata.write_h5ad(results_ctp_path + "simulated_st_" + id + ".h5ad")